In [ ]:
pip install lightning

In [ ]:
pip install torch pandas scikit-learn

In [40]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
from torch.optim import Adam 
import lightning as L 
from torch.utils.data import TensorDataset, DataLoader 

import pandas as pd 
from sklearn.model_selection import train_test_split

## Reading wine.csv dataset

In [41]:
url = './wine.csv'
df = pd.read_table(url, sep=',')

In [42]:
df.head()

,Wine,Alcohol,Malic.acid,Ash,Acl,Mg,Phenols,Flavanoids,Nonflavanoid.phenols,Proanth,Color.int,Hue,OD,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


In [43]:
df.shape

(178, 14)

In [44]:
df['Wine'].nunique()

3

In [45]:
df['Wine'].unique()

array([1, 2, 3])

In [46]:
for class_name in df['Wine'].unique():
    print(class_name, ": ", sum(df['Wine'] == class_name), sep = "")

1: 59
2: 71
3: 48


In [47]:
df[['Alcohol', 'Color.int']].head()

,Alcohol,Color.int
0,14.23,5.64
1,13.20,4.38
2,13.16,5.68
3,14.37,7.80
4,13.24,4.32


## Creating Two New DataFrames Input_Values and Label_Values

In [48]:
input_values = df[['Alcohol', 'Color.int']]
label_values = df['Wine']

print(input_values.head())
print(label_values.head())


label_values = label_values.factorize()[0]
print(label_values[:10])

   Alcohol  Color.int
0    14.23       5.64
1    13.20       4.38
2    13.16       5.68
3    14.37       7.80
4    13.24       4.32
0    1
1    1
2    1
3    1
4    1
Name: Wine, dtype: int64
[0 0 0 0 0 0 0 0 0 0]


## Issue: `KeyError` and Shape Errors After Loading Data with `header=None`

### Context

The dataset was loaded using:

```python
df = pd.read_table(url, sep=',', header=None)

This explicitly tells pandas that the file does not contain column headers. As a result:
	•	Column names are assigned as integers: 0, 1, 2, ...
	•	The first row of the file (which visually looks like headers) is treated as data
	•	Column names such as "Wine" do not exist in df.columns

However, the dataset does contain column names in the first row (e.g. Wine, Alcohol, etc.).
This mismatch caused a series of downstream issues when extracting labels and preparing them for PyTorch.

⸻

Problem 1: Column Access Fails

Later code attempted to access the label column using:

df['Wine']

This failed because:
	•	With header=None, there is no column named "Wine"
	•	"Wine" exists only as a string value in the first row, not as a column label

Depending on how the data is accessed, this results in either a KeyError or silent data misalignment.

⸻

Problem 2: One-Hot Encoding Fails with KeyError

When running:

one_hot_label_train = F.one_hot(torch.tensor(label_train)).float()

an error such as the following was raised:

KeyError: 7


⸻

Why This Happened
	•	label_train was still a pandas Series
	•	PyTorch internally iterates over the object to build a tensor
	•	pandas interprets integer access as index-based lookup, not positional access
	•	When PyTorch requested element 7, pandas attempted label_train[7]
	•	Since 7 was not an index label, pandas raised a KeyError

This error originates from pandas indexing semantics, not from PyTorch or one_hot() itself.

⸻

Root Cause Summary

There were two interacting issues:
	1.	The dataset was loaded without headers even though headers were present
	2.	pandas objects were passed directly into PyTorch without explicit conversion

Together, these caused:
	•	Incorrect column access
	•	Ambiguous indexing behavior
	•	KeyError during tensor conversion

⸻

Resolution

Step 1: Encode Labels Explicitly and Remove pandas Indexing

label_train = label_train.factorize()[0]

This:
	•	Converts labels to 0-based integers
	•	Removes pandas index semantics
	•	Produces data directly compatible with PyTorch

⸻

Remember This: Why pandas Series Can Break PyTorch Code

pandas Series and PyTorch tensors handle indexing in fundamentally different ways, which can lead to subtle and confusing errors when they are mixed directly.

In pandas, integers are treated as index labels, not positions. For example:

s = pd.Series([10, 20, 30], index=[3, 5, 7])

s[7]  # returns 30
s[1]  # KeyError, because 1 is not an index label

Even though 20 exists at position 1, pandas does not interpret s[1] as positional access.

PyTorch, on the other hand, assumes positional indexing. When creating a tensor, it internally accesses elements like s[0], s[1], s[2].
If the Series index is not [0, 1, 2], pandas raises a KeyError.

This is why passing a pandas Series directly into torch.tensor() can fail.

⸻

Rule of Thumb

pandas → encode / numpy → torch

Always remove pandas indexing semantics before passing data into PyTorch.



## Splitting into training and testing datasets

In [49]:
input_train, input_test, label_train, label_test = train_test_split(
    input_values, label_values, test_size = 0.25, stratify = label_values,
    random_state = 42)

In [50]:
input_train.shape

(133, 2)

In [51]:
input_test.shape

(45, 2)

## One-Hot Encoding

In [52]:
one_hot_lable_train = F.one_hot(torch.tensor(label_train)).type(torch.float32)

In [53]:
one_hot_lable_train[:10]

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [1., 0., 0.],
        [0., 1., 0.],
        [1., 0., 0.],
        [0., 0., 1.],
        [0., 1., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        [0., 0., 1.]])

### To avoid data leakage which often leads to model overfitting we do normalization

In [54]:
max_vals_in_input_train = input_train.max()
## Now print them out...
max_vals_in_input_train

Alcohol      14.83
Color.int    13.00
dtype: float64

In [55]:
min_vals_in_input_train = input_train.min()
## Now print them out...
min_vals_in_input_train

Alcohol      11.03
Color.int     1.28
dtype: float64

In [56]:
input_train = (input_train - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_train.head()

,Alcohol,Color.int
8,1.000000,0.334471
104,0.389474,0.141638
36,0.592105,0.283276
78,0.342105,0.180887
2,0.560526,0.375427


In [57]:
input_test = (input_test - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_test.head()

,Alcohol,Color.int
35,0.644737,0.325939
93,0.331579,0.074232
7,0.797368,0.321672
28,0.747368,0.274744
87,0.163158,0.112628


### we did one-hot encoding for label_values and normalization for input values

## Converting the DataFrame input_train into tensors

In [58]:
input_train_tensors = torch.tensor(input_train.values).type(torch.float32)
input_train_tensors[:5]

tensor([[1.0000, 0.3345],
        [0.3895, 0.1416],
        [0.5921, 0.2833],
        [0.3421, 0.1809],
        [0.5605, 0.3754]])

In [59]:
input_test_tensors = torch.tensor(input_test.values).type(torch.float32)
input_test_tensors[:5]

tensor([[0.6447, 0.3259],
        [0.3316, 0.0742],
        [0.7974, 0.3217],
        [0.7474, 0.2747],
        [0.1632, 0.1126]])

### Now that we have tensors for input_train, named input_train_tensors, and we have the one-hot encoded class values stored in tensors called label_train, we can combine them into a TensorDataset that are, in turn, turned into DataLoader.

In [60]:
train_dataset = TensorDataset(input_train_tensors, one_hot_lable_train)
train_dataloader = DataLoader(train_dataset)

## Our Neural Network with multiple inputs and outputs

In [61]:
class MultipleInsOuts(L.LightningModule):
    def __init__(self):
        super().__init__()
        L.seed_everything(seed=42)
        self.input_to_hidden = nn.Linear(in_features=2, out_features=2, bias = True)
        self.hidden_to_output = nn.Linear(in_features=2, out_features=3, bias = True)

        self.train_losses = []  # <-- store epoch losses
        self.train_accuracies = []
        self.loss = nn.MSELoss(reduction = 'sum')


    def forward(self, input):
        hidden = self.input_to_hidden(input)
        output_values = self.hidden_to_output(torch.relu(hidden))

        return output_values

    def configure_optimizers(self):
        return Adam(self.parameters(), lr = 0.001)


    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self.forward(inputs)
        loss = self.loss(outputs, labels)

        preds = torch.argmax(outputs, dim = 1)
        targets = torch.argmax(labels, dim = 1)
        acc = (preds == targets).float().mean()
        
        self.log(
            "train_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True
            )
        self.log(
            "train_acc", acc, on_step = False, on_epoch = True, prog_bar = True)

        return loss

    def on_train_epoch_end(self):
        epoch_loss = self.trainer.callback_metrics["train_loss"]
        self.train_losses.append(epoch_loss.cpu().item())

        self.train_accuracies.append(self.trainer.callback_metrics["train_acc"].cpu().item())

In [62]:
model = MultipleInsOuts()
for name, param in model.named_parameters():
    print(name, torch.round(param.data, decimals = 2))

Seed set to 42


input_to_hidden.weight tensor([[ 0.5400,  0.5900],
        [-0.1700,  0.6500]])
input_to_hidden.bias tensor([-0.1500,  0.1400])
hidden_to_output.weight tensor([[-0.3400,  0.4200],
        [ 0.6200, -0.5200],
        [ 0.6100,  0.1300]])
hidden_to_output.bias tensor([0.5200, 0.1000, 0.3400])


## Training our Neural Network

In [63]:
model = MultipleInsOuts()

Seed set to 42


In [64]:
trainer = L.Trainer(max_epochs = 10)
trainer.fit(model, train_dataloaders = train_dataloader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name             | Type    | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | input_to_hidden  | Linear  | 6      | train | 0    
1 | hidden_to_output | Linear  | 9      | train | 0    
2 | loss             | MSELoss | 0      | train | 0    
-------------------------------------------------------------
15        Trainable params
0         Non-trainable params
15        Total params
0.000     Total estimated model params size (MB)
3      

Epoch 9: 100%|█| 133/133 [00:00<00:00, 359.77it/s, v_num=15, train_loss=0.584, t

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|█| 133/133 [00:00<00:00, 353.26it/s, v_num=15, train_loss=0.584, t


In [65]:
predictions = model(input_test_tensors)

In [66]:
predictions[0:4]

tensor([[0.3493, 0.3036, 0.3342],
        [0.3368, 0.4156, 0.1373],
        [0.3386, 0.3163, 0.3609],
        [0.3348, 0.3403, 0.3249]], grad_fn=<SliceBackward0>)

In [67]:
predicted_labels = torch.argmax(predictions, dim = 1)
predicted_labels[0:4]

tensor([0, 1, 2, 1])

## Finding % of predictions correct on testing data

In [68]:
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels))/ len(predicted_labels)

tensor(0.4889)

## Imporving Prediction % by running more epochs

In [ ]:
path_to_checkpoint = trainer.checkpoint_callback.best_model_path 
trainer = L.Trainer(max_epochs = 100)
trainer.fit(model, train_dataloaders = train_dataloader, ckpt_path = path_to_checkpoint)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Restoring states from the checkpoint path at /Users/niteshkr.jha/Desktop/Neural_Network and AI/lightning_logs/version_15/checkpoints/epoch=9-step=1330.ckpt
/opt/anaconda3/envs/GoQuant/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:566: The dirpath has changed from '/Users/niteshkr.jha/Desktop/Neural_Network and AI/lightning_logs/version_15/checkpoints' to '/Users/niteshkr.jha/Desktop/Neural_Network and AI/lightning_logs/version_16/checkpoints', therefo

Epoch 84:  83%|▊| 111/133 [00:00<00:00, 288.82it/s, v_num=16, train_loss=0.309, 

In [ ]:
predictions = model(input_test_tensors)
predicted_labels = torch.argmax(predictions, dim=1)
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

In [ ]:
path_to_checkpoint = trainer.checkpoint_callback.best_model_path 
trainer = L.Trainer(max_epochs = 500)
trainer.fit(model, train_dataloaders = train_dataloader, ckpt_path = path_to_checkpoint)

In [ ]:
predictions = model(input_test_tensors)
predicted_labels = torch.argmax(predictions, dim=1)
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

In [ ]:
trainer = L.Trainer(max_epochs = 200)
trainer.fit(model, train_dataloaders = train_dataloader)

predictions = model(input_test_tensors)
predicted_labels = torch.argmax(predictions, dim=1)
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

In [ ]:
pip install matplotlib

## Plotting Graph for Loss vs Epoch

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(model.train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss vs Epoch")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(model.train_accuracies)
plt.xlabel("Epoch")
plt.ylabel("Training Accuracy")
plt.title("Training Accuracy vs Epoch")
plt.show()

### How Accuracy Is Computed (With Examples)

For each input, the model outputs one score per class.  
Example output for a single sample with three classes:

```text
[2.4, 0.8, -1.1]

Each value represents the model’s confidence for a class. The predicted class is the one with the highest score.

preds = torch.argmax(outputs, dim=1)

argmax returns the index of the largest value, converting model outputs into class indices.

Example:

[2.4, 0.8, -1.1] → class 0
[0.2, 1.9, 0.3]  → class 1


⸻

Why argmax Is Used on Labels

The labels are one-hot encoded. For example:

[0, 1, 0]

This represents class 1. To compare predictions with labels, they must be in the same format.

targets = torch.argmax(labels, dim=1)

This converts one-hot labels back into class indices.

⸻

Computing Accuracy

acc = (preds == targets).float().mean()

Step by step:
	•	preds == targets produces a boolean tensor:
[True, False, True]
	•	.float() avoids booleans by converting:
[1.0, 0.0, 1.0]
	•	.mean() averages the values:
(1 + 0 + 1) / 3 = 0.67

The result is the fraction of correct predictions, which is accuracy.

⸻

What prog_bar=True Does

self.log("train_acc", acc, prog_bar=True)

Setting prog_bar=True tells Lightning to display this metric in the training progress bar while training runs.
It does not affect optimization or gradients — it simply provides real-time feedback so you can see whether accuracy is improving during training.

⸻

Key Idea to Remember

Accuracy is just counting correct predictions, expressed as a fraction, and argmax is the bridge between raw model outputs and human-readable class labels.

